In [2]:
"""
Entity resolution, STEP 1: find candidate variant groups only.

Embeds all unique entities (source + target pooled) and finds
candidate synonym/variant groups via cosine similarity. No LLM call
yet, this just surfaces groups worth a look. Review and hand-edit
the output before running step2_validate_entity_groups.py, remove
false-positive groups, split a group, or add an entity you know is
a variant but that fell just under the similarity threshold.

Requires OPENAI_API_KEY as an environment variable (embeddings only).
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import time
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "final_resolved_triples.xlsx"  # output of apply_entity_merges.py, fully resolved
SOURCE_COL = "final_source"
TARGET_COL = "final_target"
DOI_COL = "doi"

EMBED_MODEL = "text-embedding-3-small"
SIMILARITY_THRESHOLD = 0.90   # stricter than type resolution, tune and rerun as needed
# No cap on candidates per entity: every neighbor above SIMILARITY_THRESHOLD is kept,
# regardless of how many that is. A fixed cap here would silently truncate genuinely
# similar entities once a family exceeds the cap (e.g. DPPH/ABTS families with 7+
# true members), fragmenting one real group into several overlapping ones.

OUTPUT_CANDIDATES_XLSX = "entity_candidate_groups_for_review2.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------------------------------------------------------------
# LOAD AND POOL ALL UNIQUE ENTITIES
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)

source_long = df[[SOURCE_COL, DOI_COL]].rename(columns={SOURCE_COL: "entity"})
target_long = df[[TARGET_COL, DOI_COL]].rename(columns={TARGET_COL: "entity"})
entity_long = pd.concat([source_long, target_long])
entity_long["entity"] = entity_long["entity"].astype(str).str.strip()

entities = sorted(entity_long["entity"].unique())
entity_doi_counts = entity_long.groupby("entity")[DOI_COL].nunique().to_dict()
entity_dois = entity_long.groupby("entity")[DOI_COL].apply(lambda s: sorted(s.unique())).to_dict()

print(f"Unique entities (source + target pooled): {len(entities)}")

# ---------------------------------------------------------------
# EMBED
# ---------------------------------------------------------------
def embed_batch(texts, batch_size=200):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in resp.data])
        time.sleep(0.2)
    return np.array(vectors)


embeddings = embed_batch(entities)
print(f"Embedded {embeddings.shape[0]} entities into {embeddings.shape[1]}-dim vectors")

# ---------------------------------------------------------------
# FIND CANDIDATE VARIANT GROUPS VIA PAIRWISE SIMILARITY
# ---------------------------------------------------------------
sim_matrix = cosine_similarity(embeddings)
np.fill_diagonal(sim_matrix, -1)

candidate_groups = []
seen = set()
group_id = 0

for i, entity in enumerate(entities):
    if entity in seen:
        continue
    # keep every neighbor above SIMILARITY_THRESHOLD, no top-K cap
    match_idx = np.where(sim_matrix[i] >= SIMILARITY_THRESHOLD)[0]
    matches = [(entities[j], sim_matrix[i, j]) for j in match_idx]
    if matches:
        group = [entity] + [m[0] for m in matches]
        max_sim = max(m[1] for m in matches)
        # combined unique DOI count across all members, built from the entity_dois
        # dict (real lists) rather than string joins, low value here is a signal
        # the "variant" might just be one paper's inconsistent phrasing
        combined_doi_count = len(set().union(*[set(entity_dois[m]) for m in group]))
        for member in group:
            candidate_groups.append({
                "group_id": group_id,
                "entity": member,
                "max_similarity_in_group": round(max_sim, 3),
                "entity_doc_frequency": entity_doi_counts.get(member, 0),
                "dois": "; ".join(entity_dois[member]),
                "group_combined_doc_frequency": combined_doi_count,
                "notes": "",
            })
        seen.update(group)
        group_id += 1

candidates_df = pd.DataFrame(candidate_groups)
print(f"Found {candidates_df['group_id'].nunique()} candidate variant groups above similarity {SIMILARITY_THRESHOLD}")

# ---------------------------------------------------------------
# SAVE FOR MANUAL REVIEW
# ---------------------------------------------------------------
candidates_df = candidates_df.sort_values(["group_id", "entity"]).reset_index(drop=True)
candidates_df.to_excel(OUTPUT_CANDIDATES_XLSX, index=False)

print(f"\nSaved candidate groups to {OUTPUT_CANDIDATES_XLSX}")
print("Review and edit group_id values as needed (remove false positives, merge, split, add missed variants).")
print("Then run step2_validate_entity_groups.py on this file.")

Unique entities (source + target pooled): 3100
Embedded 3100 entities into 1536-dim vectors
Found 342 candidate variant groups above similarity 0.9

Saved candidate groups to entity_candidate_groups_for_review2.xlsx
Review and edit group_id values as needed (remove false positives, merge, split, add missed variants).
Then run step2_validate_entity_groups.py on this file.


In [1]:
"""
Entity resolution, STEP 1: find candidate variant groups only.

Embeds all unique entities (source + target pooled) and finds
candidate synonym/variant groups via cosine similarity. No LLM call
yet, this just surfaces groups worth a look. Review and hand-edit
the output before running step2_validate_entity_groups.py, remove
false-positive groups, split a group, or add an entity you know is
a variant but that fell just under the similarity threshold.

Requires OPENAI_API_KEY as an environment variable (embeddings only).
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import time
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "final_resolved_triples.xlsx"  # output of apply_entity_merges.py, fully resolved
SOURCE_COL = "final_source"
TARGET_COL = "final_target"
DOI_COL = "doi"

EMBED_MODEL = "text-embedding-3-small"
SIMILARITY_THRESHOLD = 0.90   # confirmed via direct comparison: 0.85 produced false positives
# (e.g. grouped "alcalase hydrolysis" with "fermentation"), 0.90 did not on this corpus
# No cap on candidates per entity: every neighbor above SIMILARITY_THRESHOLD is kept,
# regardless of how many that is. A fixed cap here would silently truncate genuinely
# similar entities once a family exceeds the cap (e.g. DPPH/ABTS families with 7+
# true members), fragmenting one real group into several overlapping ones.

OUTPUT_CANDIDATES_XLSX = "entity_candidate_groups_for_review.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------------------------------------------------------------
# LOAD AND POOL ALL UNIQUE ENTITIES
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)

source_long = df[[SOURCE_COL, DOI_COL]].rename(columns={SOURCE_COL: "entity"})
target_long = df[[TARGET_COL, DOI_COL]].rename(columns={TARGET_COL: "entity"})
entity_long = pd.concat([source_long, target_long])
entity_long["entity"] = entity_long["entity"].astype(str).str.strip()

entities = sorted(entity_long["entity"].unique())
entity_doi_counts = entity_long.groupby("entity")[DOI_COL].nunique().to_dict()
entity_dois = entity_long.groupby("entity")[DOI_COL].apply(lambda s: sorted(s.unique())).to_dict()

print(f"Unique entities (source + target pooled): {len(entities)}")

# ---------------------------------------------------------------
# EMBED
# ---------------------------------------------------------------
def embed_batch(texts, batch_size=200):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in resp.data])
        time.sleep(0.2)
    return np.array(vectors)


embeddings = embed_batch(entities)
print(f"Embedded {embeddings.shape[0]} entities into {embeddings.shape[1]}-dim vectors")

# ---------------------------------------------------------------
# FIND CANDIDATE VARIANT GROUPS VIA PAIRWISE SIMILARITY
# ---------------------------------------------------------------
sim_matrix = cosine_similarity(embeddings)
np.fill_diagonal(sim_matrix, -1)

candidate_groups = []
seen = set()
group_id = 0

for i, entity in enumerate(entities):
    if entity in seen:
        continue
    # keep every neighbor above SIMILARITY_THRESHOLD, no top-K cap.
    # CRITICAL: also exclude neighbors already claimed by an earlier
    # group. Without this, an entity that's a strong match of TWO
    # different anchors (processed at different points in the loop)
    # ends up duplicated across two separate groups, confirmed on real
    # data: "sorghum gluten meal protein" appeared in both group 270
    # and group 289 before this fix, since only the ANCHOR was checked
    # against `seen`, not the matched neighbors.
    match_idx = np.where(sim_matrix[i] >= SIMILARITY_THRESHOLD)[0]
    matches = [(entities[j], sim_matrix[i, j]) for j in match_idx if entities[j] not in seen]
    if matches:
        group = [entity] + [m[0] for m in matches]
        max_sim = max(m[1] for m in matches)
        # combined unique DOI count across all members, built from the entity_dois
        # dict (real lists) rather than string joins, low value here is a signal
        # the "variant" might just be one paper's inconsistent phrasing
        combined_doi_count = len(set().union(*[set(entity_dois[m]) for m in group]))
        for member in group:
            candidate_groups.append({
                "group_id": group_id,
                "entity": member,
                "max_similarity_in_group": round(max_sim, 3),
                "entity_doc_frequency": entity_doi_counts.get(member, 0),
                "dois": "; ".join(entity_dois[member]),
                "group_combined_doc_frequency": combined_doi_count,
                "notes": "",
            })
        seen.update(group)
        group_id += 1

candidates_df = pd.DataFrame(candidate_groups)
print(f"Found {candidates_df['group_id'].nunique()} candidate variant groups above similarity {SIMILARITY_THRESHOLD}")

# ---------------------------------------------------------------
# SAVE FOR MANUAL REVIEW
# ---------------------------------------------------------------
candidates_df = candidates_df.sort_values(["group_id", "entity"]).reset_index(drop=True)
candidates_df.to_excel(OUTPUT_CANDIDATES_XLSX, index=False)

print(f"\nSaved candidate groups to {OUTPUT_CANDIDATES_XLSX}")
print("Review and edit group_id values as needed (remove false positives, merge, split, add missed variants).")
print("Then run step2_validate_entity_groups.py on this file.")

Unique entities (source + target pooled): 3100
Embedded 3100 entities into 1536-dim vectors
Found 297 candidate variant groups above similarity 0.9

Saved candidate groups to entity_candidate_groups_for_review.xlsx
Review and edit group_id values as needed (remove false positives, merge, split, add missed variants).
Then run step2_validate_entity_groups.py on this file.
